# VoltRelay Energy — What is really driving service quality, retention and margin?

**Gradient Learnings Data Analytics Hackathon** · Battery-swap network, 6 Indian cities, Jan 2024 – Jun 2025

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andringodson/Hackathon-DataAnalytics/blob/main/notebooks/VoltRelay_Analysis.ipynb)

This notebook runs top to bottom in Google Colab. The first code cell downloads the eight organiser CSVs (~830 MB) from Google Drive.

| Section | Contents |
|---|---|
| 1 | Setup and data loading |
| 2 | Data understanding |
| 3 | Data cleaning — every documented quality issue tested, plus three undocumented ones |
| 4 | Metric definitions and unit-economics model |
| 5–10 | The six Core Questions |
| 11 | Optional deep dives: budget decision, expansion waves, partner value, ticket text, anomalies |
| 12 | Key findings and recommendations |
| 13 | Export of aggregated tables for the dashboard |

## 1. Setup and data loading

In [ ]:
import os, sys, subprocess, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
try:
    import gdown  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

DATA_DIR = Path(os.environ.get("VOLTRELAY_DATA", "data/raw"))
FIG_DIR = Path(os.environ.get("VOLTRELAY_FIGS", "figures"))
OUT_DIR = Path(os.environ.get("VOLTRELAY_OUT", "processed"))
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

TABLES = ["swap_events", "station_hourly_status", "riders", "batteries", "support_tickets", "stations", "city_daily_context", "fleet_partners"]
DRIVE_FOLDER = "https://drive.google.com/drive/folders/1qsjGWrgmEvOaf2Is2fh7RgVEkKd1V3df"


def find(name):
    hits = sorted(DATA_DIR.rglob(f"{name}.csv*"))
    return hits[0] if hits else None


if any(find(t) is None for t in TABLES):
    import gdown
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(DRIVE_FOLDER, output=str(DATA_DIR), quiet=True, remaining_ok=True)
missing = [t for t in TABLES if find(t) is None]
assert not missing, f"Missing files: {missing}"
print({t: f"{find(t).stat().st_size / 1e6:,.1f} MB" for t in TABLES})

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
PAL = {"primary": "#1f5f8b", "accent": "#e07a1f", "bad": "#c0392b", "good": "#2e8b57", "muted": "#8c8c8c"}
GEN_PAL = {"Gen1": "#c0392b", "Gen2": "#1f5f8b", "Gen3": "#2e8b57"}
plt.rcParams.update({"figure.dpi": 90, "savefig.dpi": 130, "axes.titleweight": "bold", "axes.titlesize": 12})


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()


def pct(ax, axis="y", decimals=0):
    fmt = mtick.PercentFormatter(1.0, decimals=decimals)
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(fmt)

In [ ]:
cat = "category"
se = pd.read_csv(find("swap_events"), parse_dates=["event_ts"], dtype={
    "rider_id": cat, "station_id": cat, "event_type": cat, "battery_in_id": cat, "battery_out_id": cat,
    "tariff_code": cat, "payment_mode": cat, "station_firmware": cat, "sync_mode": cat})
hs = pd.read_csv(find("station_hourly_status"), parse_dates=["hour_start"], dtype={"station_id": cat, "telemetry_status": cat})
rd = pd.read_csv(find("riders"), parse_dates=["signup_date"])
bt = pd.read_csv(find("batteries"), parse_dates=["manufacture_date", "commission_date", "retired_date"])
tk = pd.read_csv(find("support_tickets"), parse_dates=["created_ts"])
st = pd.read_csv(find("stations"), parse_dates=["commissioned_date", "decommissioned_date", "firmware_updated_date", "competitor_within_1_5km_since"])
cx = pd.read_csv(find("city_daily_context"), parse_dates=["date"])
fp = pd.read_csv(find("fleet_partners"), parse_dates=["contract_start_date", "amendment_date"])
raw = {"swap_events": se, "station_hourly_status": hs, "riders": rd, "batteries": bt, "support_tickets": tk,
       "stations": st, "city_daily_context": cx, "fleet_partners": fp}
pd.DataFrame({k: {"rows": len(v), "columns": v.shape[1]} for k, v in raw.items()}).T

## 2. Data understanding

The core fact table is `swap_events` — one row per swap *attempt*, completed or not. Everything else hangs off it:

```
fleet_partners -> riders -> swap_events -> stations -> station_hourly_status
                                 |   \-> batteries (battery_in_id, battery_out_id)
                                 \-> support_tickets -> stations / batteries (optional)
stations.city + date -> city_daily_context
```

In [ ]:
se.head(3).T

In [ ]:
print("Event outcomes:")
display(se.event_type.value_counts(normalize=True).rename("share").to_frame())
print("Tariff codes:", se.tariff_code.value_counts().to_dict())
print("Payment modes:", se.payment_mode.value_counts().to_dict())
print("Date range:", se.event_ts.min(), "->", se.event_ts.max())

**Referential integrity.** Every foreign key resolves:

In [ ]:
checks = {
    "swap_events.rider_id in riders": se.rider_id.isin(rd.rider_id).mean(),
    "swap_events.station_id in stations": se.station_id.isin(st.station_id).mean(),
    "swap_events.battery_out_id in batteries": se.battery_out_id.dropna().isin(bt.battery_id).mean(),
    "support_tickets.rider_id in riders": tk.rider_id.isin(rd.rider_id).mean(),
    "riders.partner_id in fleet_partners": rd.partner_id.dropna().isin(fp.partner_id).mean(),
}
pd.Series(checks, name="share matched").to_frame()

**Which fields are populated for which outcome.** Pricing, energy and outgoing-battery fields exist only for completed swaps; non-completed attempts carry ₹0. Revenue and unit-economics analysis therefore uses completed swaps only, while service-quality analysis uses all attempts.

In [ ]:
se.groupby("event_type", observed=True)[["battery_in_id", "battery_out_id", "km_since_last_swap", "energy_to_recharge_kwh", "amount_charged_inr"]].agg(
    lambda s: s.notna().mean() if s.name != "amount_charged_inr" else s.mean()).rename(columns={"amount_charged_inr": "mean ₹ charged"})

## 3. Data cleaning

Each documented data-quality issue is **detected with evidence, then handled deliberately**. The analysis also found three undocumented issues (3.8).

### 3.1 Test stations — excluded, but they are not "small"
The brief describes `STN-TST-*` as inactive test stations with a few zero-value transactions. In this export they are anything but.

In [ ]:
tst_ids = st.loc[st.station_id.str.startswith("STN-TST"), "station_id"].tolist()
tst = se[se.station_id.isin(tst_ids)]
real_avg = se[~se.station_id.isin(tst_ids)].groupby("station_id", observed=True).size().mean()
display(tst.groupby("station_id", observed=True).agg(events=("event_id", "size"), revenue_inr=("amount_charged_inr", "sum"),
        zero_value_share=("amount_charged_inr", lambda s: (s == 0).mean()), first=("event_ts", "min"), last=("event_ts", "max")))
print(f"Average real station: {real_avg:,.0f} events")

**Decision:** exclude both test stations (62K events, ₹3.9M) from every network metric, as the brief instructs. **Flag:** the test stations handle more traffic than an average real station and book real revenue for all 18 months. Either they are live sites that were mislabelled, or revenue is being recorded on test infrastructure. Finance should reconcile this.

In [ ]:
se = se[~se.station_id.isin(tst_ids)].copy()
hs = hs[~hs.station_id.isin(tst_ids)].copy()
tk = tk[~tk.station_id.isin(tst_ids)].copy()
st = st[~st.station_id.isin(tst_ids)].copy()
for df in (se, hs):
    df["station_id"] = df.station_id.cat.remove_unused_categories()
print("Stations remaining:", st.shape[0])

### 3.2 Firmware timezone bug (v3.2.0, 10 Mar – 14 Apr 2025)
The strongest proof: the **PEAK tariff only applies at 12–14h and 19–22h**, yet affected events show PEAK prices charged at 06–08h and 13–16h. That is exactly 5 h 30 m earlier than the true time (IST vs UTC).

In [ ]:
bug = (se.station_firmware == "v3.2.0") & (se.event_ts >= "2025-03-10") & (se.event_ts < "2025-04-15")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
for ax, (label, mask) in zip(axes, [("v3.2.0 during bug window", bug), ("All other events", ~bug)]):
    se.loc[mask, "event_ts"].dt.hour.value_counts(normalize=True).sort_index().plot.bar(ax=ax, color=PAL["bad"] if "bug" in label else PAL["primary"])
    ax.set_title(label); ax.set_xlabel("logged hour of day"); pct(ax)
axes[0].set_ylabel("share of events")
save(fig, "03_firmware_bug_hours")
peak_hours_logged = se.loc[bug & (se.tariff_code == "PEAK"), "event_ts"].dt.hour.value_counts().sort_index()
print("PEAK-tariff events by *logged* hour in the bug window:", peak_hours_logged[peak_hours_logged > 0].to_dict())

In [ ]:
se["ts_corrected"] = bug
se.loc[bug, "event_ts"] = se.loc[bug, "event_ts"] + pd.Timedelta(hours=5, minutes=30)
chk = se.loc[bug & (se.tariff_code == "PEAK"), "event_ts"].dt.hour.value_counts()
print(f"Corrected {bug.sum():,} timestamps. PEAK events now fall in hours: {sorted(chk[chk > 0].index.tolist())}")

### 3.3 Offline-sync near-duplicates — tested, none present
Definition from the brief: same rider, station and batteries, a few seconds to two minutes apart, concentrated in `offline_batch` rows at poor-connectivity stations.

In [ ]:
s = se.sort_values(["rider_id", "station_id", "event_ts"])
p = s.shift()
gap = (s.event_ts - p.event_ts).dt.total_seconds()
same_rs = s.rider_id.astype(str).eq(p.rider_id.astype(str)) & s.station_id.astype(str).eq(p.station_id.astype(str))
same_batt = (s.battery_in_id.astype(str).eq(p.battery_in_id.astype(str)) & s.battery_out_id.astype(str).eq(p.battery_out_id.astype(str)))
exact_dups = same_rs & same_batt & (gap <= 120)
close = same_rs & (gap <= 120)
print(f"Exact near-duplicates (same rider+station+batteries, <=2 min): {exact_dups.sum()}")
print(f"Same rider+station within 2 min (any batteries): {close.sum():,}")
print(f"  offline_batch share among them: {(s.sync_mode[close] == 'offline_batch').mean():.1%}  vs network baseline {(se.sync_mode == 'offline_batch').mean():.1%}")
conn = se[["station_id", "sync_mode"]].merge(st[["station_id", "connectivity_tier"]], on="station_id")
print(pd.crosstab(conn.connectivity_tier, conn.sync_mode, normalize="index").round(3))
del s, p, gap

**Decision:** no rows removed. Close same-rider pairs always involve *different* battery packs, and `offline_batch` is no more common among them than in the network overall. They are genuine back-to-back attempts, not sync retries. The check stays in the pipeline so a future export with real duplicates would be caught. `offline_batch` itself does behave as documented: it only occurs at fair and poor-connectivity stations.

### 3.4 Distance and charge-reading outliers

In [ ]:
km = se.km_since_last_swap
print(km.describe(percentiles=[.001, .01, .99, .999]).round(1).to_dict())
km_bad = (km < 0) | (km > 250)
print(f"Negative (odometer reset): {(km < 0).sum():,} | >250 km (implausible for a 2.1/4.8 kWh pack): {(km > 250).sum():,}")
se["km_valid"] = km.where(~km_bad)
for c in ["soc_in_pct", "soh_in_pct", "soc_out_pct", "soh_out_pct"]:
    n = (se[c] > 100).sum()
    se[c] = se[c].clip(upper=100)
    print(f"{c}: {n:,} readings >100% clipped to 100 (sensor drift)")

### 3.5 Inconsistent city spellings in `riders.home_city`

In [ ]:
city_map = {"bengaluru": "Bengaluru", "bangalore": "Bengaluru", "blr": "Bengaluru",
            "delhi ncr": "Delhi NCR", "delhi": "Delhi NCR", "new delhi": "Delhi NCR", "gurgaon": "Delhi NCR",
            "hyderabad": "Hyderabad", "hyd": "Hyderabad", "pune": "Pune", "pun": "Pune",
            "mumbai": "Mumbai", "bombay": "Mumbai", "mum": "Mumbai", "jaipur": "Jaipur", "jai": "Jaipur"}
print("Raw variants:", rd.home_city.nunique())
rd["home_city"] = rd.home_city.str.strip().str.lower().map(city_map)
assert rd.home_city.notna().all()
rd.home_city.value_counts().to_frame().T

### 3.6 Missing telemetry — kept as missing, never zero
Blank telemetry fields line up exactly with `telemetry_status` ∈ {partial, missing}, concentrated at poor-connectivity stations. Hourly aggregates use only `ok` rows, so a blank is never read as "0 charged packs" (which would fake a stockout).

In [ ]:
hs = hs.merge(st[["station_id", "connectivity_tier", "charger_generation", "city", "expansion_wave"]], on="station_id")
display(pd.crosstab(hs.connectivity_tier, hs.telemetry_status, normalize="index").round(3))
print("Null charged_2w_min where status == ok:", hs.loc[hs.telemetry_status == "ok", "charged_2w_min"].isna().sum())

### 3.7 CSAT is missing not-at-random
CSAT exists **only for resolved tickets**, and half as often when resolution takes more than 24 h. A naive average therefore describes the happiest subset only. We report CSAT with its coverage and never as a stand-alone satisfaction metric.

In [ ]:
display(tk.groupby("resolution_status").csat_score.agg(coverage=lambda s: s.notna().mean(), mean="mean"))
display(tk.groupby(pd.cut(tk.resolution_hours, [0, 4, 12, 24, 48, 1000]), observed=True).csat_score.agg(coverage=lambda s: s.notna().mean(), mean="mean"))

### 3.8 Three undocumented issues found during profiling
1. **`payment_mode` is always `partner_invoice`**, even for independent pay-as-you-go riders. It carries no information, so rider segments come from `riders.plan_type` and `partner_id`.
2. **Tariffs `PREPAID` and `PROMO_FREE` never occur.** Prepaid riders are billed on `STD`/`PEAK`/`OFFPEAK`.
3. **Battery IDs are not physically consistent across events.** The same pack is "issued" in different cities minutes apart (below). Batteries are therefore analysed as *cohorts* (supplier, lot, SoH at swap time), not by tracing individual packs through the event log.

In [ ]:
b = se.dropna(subset=["battery_out_id"]).sort_values(["battery_out_id", "event_ts"])[["battery_out_id", "event_ts", "station_id", "rider_id"]]
bp = b.shift()
impossible = b.battery_out_id.astype(str).eq(bp.battery_out_id.astype(str)) & ((b.event_ts - bp.event_ts).dt.total_seconds() < 900) & b.station_id.astype(str).ne(bp.station_id.astype(str))
print(f"Same pack issued at two different stations <15 min apart: {impossible.sum():,} times")
display(pd.concat([bp[impossible].head(2), b[impossible].head(2)]).sort_values(["battery_out_id", "event_ts"]))
del b, bp

### 3.9 Cleaning summary

In [ ]:
pd.DataFrame([
    ["Test stations STN-TST-01/02", "62,031 events, ₹3.9M", "Excluded; flagged as anomaly"],
    ["Firmware v3.2.0 timezone bug", f"{bug.sum():,} events", "+5h30m correction"],
    ["Offline-sync near-duplicates", "0 found", "Check retained; nothing removed"],
    ["km_since_last_swap <0 or >250", f"{km_bad.sum():,} values", "Set to missing for range analysis"],
    ["SoC/SoH > 100%", "~13K readings", "Clipped to 100"],
    ["home_city spellings", "21 variants", "Mapped to 6 cities"],
    ["Missing telemetry", "46K hours", "Excluded from averages, never zero-filled"],
    ["CSAT MNAR", "66% missing", "Reported with coverage only"],
    ["Ticket categories", "'other' = vehicle complaints", "Re-classified with text rules (§11.4)"],
], columns=["Issue", "Scale", "Treatment"])

## 4. Metric definitions and unit-economics model

| Metric | Definition |
|---|---|
| **Attempt** | Any `swap_events` row |
| **Completed swap** | `event_type == swap_completed` |
| **Service failure** | `failed_no_charged_battery`, `failed_system_error` or `abandoned_queue`. `cancelled_by_rider` is excluded: those riders leave within 90 s, before the station has failed them. |
| **Failure rate** | Service failures ÷ attempts |
| **Revenue** | `amount_charged_inr` on completed swaps (non-completed attempts are ₹0) |
| **Contribution before wear (CM1)** | Revenue − energy cost − station fixed cost |
| **Contribution after wear (CM2)** | CM1 − battery wear cost |

**Cost model (per completed swap).** The brief names the three costs to include: energy, battery wear and station costs.
* **Energy** = `energy_to_recharge_kwh` × the station's `grid_tariff_inr_kwh`.
* **Station fixed cost** = the station's (monthly rent + maintenance) ÷ its completed swaps that month.
* **Battery wear** = pack purchase cost × the share of its useful life (initial SoH → 70% retirement threshold) consumed per swap. SoH lost *inside* the observation window is estimated by prorating lifetime loss by days in service within the window, then divided by the number of times the pack was issued. Fast-degrading packs therefore carry a proportionally higher wear cost.

Battery wear rests on the most assumptions, so every margin chart shows CM1 and CM2 side by side. The conclusions below hold under both.

In [ ]:
se = se.merge(st[["station_id", "city", "zone", "grid_tariff_inr_kwh", "charger_generation", "location_type", "expansion_wave",
                  "connectivity_tier", "host_type", "monthly_rent_inr", "monthly_maintenance_inr"]], on="station_id", how="left")
se = se.merge(rd[["rider_id", "vehicle_class", "partner_id", "plan_type"]], on="rider_id", how="left")
se["date"] = se.event_ts.dt.normalize()
se["month"] = se.event_ts.dt.to_period("M").dt.to_timestamp()
se["hour"] = se.event_ts.dt.hour
se["completed"] = se.event_type.eq("swap_completed")
se["service_failure"] = se.event_type.isin(["failed_no_charged_battery", "failed_system_error", "abandoned_queue"])
se["no_charge"] = se.event_type.eq("failed_no_charged_battery")

issues = se.loc[se.completed, "battery_out_id"].astype(str).value_counts().rename("issues")
bt = bt.merge(issues, left_on="battery_id", right_index=True, how="left")
bt["soh_loss"] = (bt.initial_soh_pct - bt.current_soh_pct).clip(lower=0)
end = bt.retired_date.fillna(pd.Timestamp("2025-06-30"))
life_days = (end - bt.commission_date).dt.days.clip(lower=1)
window_days = (end - bt.commission_date.clip(lower=pd.Timestamp("2024-01-01"))).dt.days.clip(lower=0)
bt["soh_loss_window"] = bt.soh_loss * window_days / life_days
bt["soh_loss_per_100_swaps"] = 100 * bt.soh_loss_window / bt.issues
bt["wear_inr_per_swap"] = bt.purchase_cost_inr * (bt.soh_loss_window / (bt.initial_soh_pct - 70)) / bt.issues

wear = bt.set_index("battery_id").wear_inr_per_swap
se["wear_inr"] = se.battery_out_id.astype(str).map(wear).where(se.completed)
se["energy_cost_inr"] = se.energy_to_recharge_kwh * se.grid_tariff_inr_kwh
sm = se[se.completed].groupby(["station_id", "month"], observed=True).size().rename("n").reset_index()
sm = sm.merge(st[["station_id", "monthly_rent_inr", "monthly_maintenance_inr"]], on="station_id")
sm["fixed_per_swap"] = (sm.monthly_rent_inr + sm.monthly_maintenance_inr) / sm.n
se = se.merge(sm[["station_id", "month", "fixed_per_swap"]], on=["station_id", "month"], how="left")
se.loc[~se.completed, "fixed_per_swap"] = np.nan
se["cm1"] = (se.amount_charged_inr - se.energy_cost_inr - se.fixed_per_swap).where(se.completed)
se["cm2"] = (se.cm1 - se.wear_inr).where(se.completed)
comp = se[se.completed]
comp[["amount_charged_inr", "energy_cost_inr", "fixed_per_swap", "wear_inr", "cm1", "cm2"]].describe().T[["mean", "25%", "50%", "75%"]]

## 5. Core Question 1 — Network performance over time
*How have completed swaps, revenue, failure rates and contribution margin per swap trended? Do they tell the same story?*

In [ ]:
m = se.groupby("month").agg(attempts=("event_id", "size"), completed=("completed", "sum"), failures=("service_failure", "sum"),
                            revenue=("amount_charged_inr", "sum"), stations=("station_id", "nunique"), riders=("rider_id", "nunique"))
m["failure_rate"] = m.failures / m.attempts
m = m.join(comp.groupby("month")[["amount_charged_inr", "energy_cost_inr", "fixed_per_swap", "wear_inr", "cm1", "cm2"]].mean().add_prefix("per_swap_"))
monthly = m.copy()
m[["attempts", "completed", "failures", "failure_rate", "revenue", "stations", "per_swap_amount_charged_inr", "per_swap_cm1", "per_swap_cm2"]]

In [ ]:
first, last = m.iloc[:3].mean(), m.iloc[-3:].mean()
growth = pd.DataFrame({"Q1 2024 (monthly avg)": first, "Q2 2025 (monthly avg)": last})
growth["change"] = growth.iloc[:, 1] / growth.iloc[:, 0] - 1
growth.loc[["completed", "revenue", "failures", "failure_rate", "riders", "stations", "per_swap_amount_charged_inr", "per_swap_cm1", "per_swap_cm2"]]

In [ ]:
EVENTS = [("2024-07-01", "① price rise"), ("2024-08-01", "② Kyron lots KY-2407..09"), ("2024-10-01", "③ peak pilot"), ("2024-11-01", "④ ZipDrop 12%→28%")]


def mark(ax, events=EVENTS, top=0.98, step=0.075):
    tr = ax.get_xaxis_transform()
    for i, (d, lbl) in enumerate(events):
        ax.axvline(pd.Timestamp(d), color=PAL["muted"], ls=":", lw=1)
        ax.text(pd.Timestamp(d), top - i * step, " " + lbl, transform=tr, fontsize=8, color="#444", va="top", ha="left",
                bbox=dict(facecolor="white", edgecolor="none", alpha=.8, pad=1))


fig, axes = plt.subplots(2, 2, figsize=(15, 8.5))
idx = m[["completed", "revenue", "failures"]] / m[["completed", "revenue", "failures"]].iloc[0] * 100
ax = axes[0, 0]
idx.plot(ax=ax, color=[PAL["primary"], PAL["good"], PAL["bad"]], lw=2.2)
ax.set_title("Growth index (Jan 2024 = 100): failures grew fastest"); ax.set_xlabel("")
ax = axes[0, 1]
ax.plot(m.index, m.failure_rate, color=PAL["bad"], lw=2.2, marker="o", ms=4)
ax.set_title("Service-failure rate: flat ~4% except two summer spikes"); pct(ax, decimals=0)
ax = axes[1, 0]
ax.bar(m.index, m.revenue / 1e6, width=20, color=PAL["good"])
ax.set_title("Monthly revenue (₹ million)")
ax = axes[1, 1]
ax.plot(m.index, m.per_swap_cm1, color=PAL["primary"], lw=2.2, marker="o", ms=4, label="CM1: before battery wear")
ax.plot(m.index, m.per_swap_cm2, color=PAL["accent"], lw=2.2, marker="o", ms=4, label="CM2: after battery wear")
ax.axhline(0, color="black", lw=.8)
ax.set_title("Contribution margin per completed swap (₹)"); ax.legend(loc="center left", fontsize=8); mark(ax)
fig.tight_layout()
save(fig, "05_network_trends")

In [ ]:
comp_cols = ["per_swap_amount_charged_inr", "per_swap_energy_cost_inr", "per_swap_fixed_per_swap", "per_swap_wear_inr"]
w = m[comp_cols].rename(columns=dict(zip(comp_cols, ["Revenue", "Energy", "Station fixed", "Battery wear"])))
fig, ax = plt.subplots(figsize=(13, 4.2))
ax.stackplot(w.index, w["Energy"], w["Station fixed"], w["Battery wear"], labels=["Energy", "Station fixed", "Battery wear"],
             colors=["#f2c14e", "#9aa5b1", PAL["accent"]], alpha=.85)
ax.plot(w.index, w["Revenue"], color=PAL["good"], lw=2.5, label="Revenue per swap")
ax.set_title("Per-swap revenue vs cost stack (₹): the price rise was absorbed by battery wear within two months")
ax.legend(loc="upper left", ncol=2, fontsize=8); ax.set_ylim(0, 125); mark(ax)
save(fig, "05_cost_stack")
w.round(2).iloc[[0, 5, 6, 8, 9, 10, 17]]

### Answer to Q1
**No — the metrics tell four different stories.**
* **Volume and revenue**: completed swaps grew **2.8×** and revenue **3.0×** between Q1 2024 and Q2 2025, as stations grew from 90 to 150 and monthly active riders grew 2.6×.
* **Failures grew faster than volume.** Monthly service failures rose **5.7×** (vs 2.8× for completions). The rate is not drifting up steadily: it sits at ~4% most of the year and **spikes every summer**, to 11.9% in May 2024 and 9.5% in May 2025. Growth simply puts more swaps into those summer months.
* **Margin moved against revenue.** The July 2024 base-price rise lifted revenue per swap by ~₹5 and briefly more than halved the per-swap loss (CM2 −₹14 → −₹7). From September 2024, **battery wear per swap jumped ~₹16 (+44%)** and the ZipDrop renegotiation (Nov 2024) added ~₹2 of network-wide discount per swap. Together these wiped out the price rise, and CM2 has sat near −₹16 to −₹20 since.
* Energy and station fixed costs per swap actually **fell** (better station utilisation; lower energy per swap as packs age). Battery wear is the cost that moved the margin.

## 6. Core Question 2 — Service failures and customer experience
*How do queue wait, failed and abandoned swaps relate to station, hour, season and vehicle class? Is service quality even, or concentrated?*

In [ ]:
fail_mix = pd.crosstab(se.month, se.event_type, normalize="index")
fig, ax = plt.subplots(figsize=(13, 3.8))
fail_mix.drop(columns="swap_completed").plot.area(ax=ax, color=["#e07a1f", "#9aa5b1", "#c0392b", "#6c3483"], alpha=.85)
ax.set_title("Non-completed attempts by type: summer spikes are stockouts ('no charged battery') plus the abandonment they cause")
pct(ax, decimals=0); ax.set_xlabel(""); ax.legend(fontsize=8, loc="upper center", ncol=4)
save(fig, "06_failure_mix")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
h = se.groupby("hour").agg(fail=("service_failure", "mean"), attempts=("event_id", "size"))
ax = axes[0]
ax.bar(h.index, h.attempts / 1e3, color="#d5dde5")
ax2 = ax.twinx(); ax2.plot(h.index, h.fail, color=PAL["bad"], marker="o", lw=2); pct(ax2, decimals=1); ax2.grid(False)
ax.set_title("By hour: evening peak fails most"); ax.set_xlabel("hour (corrected)"); ax.set_ylabel("attempts (000s)"); ax2.set_ylabel("failure rate")
mv = se.pivot_table(index="month", columns="vehicle_class", values="service_failure", aggfunc="mean")
ax = axes[1]
mv.plot(ax=ax, color=[PAL["primary"], PAL["accent"]], lw=2.2, marker="o", ms=3)
ax.set_title("By vehicle class: 3W fails 2× as often"); pct(ax, decimals=0); ax.set_xlabel("")
city_m = se.pivot_table(index="city", columns="month", values="service_failure", aggfunc="mean")
ax = axes[2]
sns.heatmap(city_m, ax=ax, cmap="Reds", cbar_kws={"format": mtick.PercentFormatter(1.0, decimals=0)}, xticklabels=3)
ax.set_xticklabels([pd.Timestamp(t.get_text()).strftime("%b-%y") for t in ax.get_xticklabels()], rotation=45)
ax.set_title("By city × month: summer spikes hit Delhi, Jaipur, Hyderabad"); ax.set_xlabel(""); ax.set_ylabel("")
fig.tight_layout()
save(fig, "06_failure_hour_class_city")

In [ ]:
se["season"] = se.month.dt.month.map({12: "Winter (Dec-Feb)", 1: "Winter (Dec-Feb)", 2: "Winter (Dec-Feb)", 3: "Summer (Mar-May)", 4: "Summer (Mar-May)",
                                      5: "Summer (Mar-May)", 6: "Monsoon (Jun-Sep)", 7: "Monsoon (Jun-Sep)", 8: "Monsoon (Jun-Sep)", 9: "Monsoon (Jun-Sep)",
                                      10: "Post-monsoon (Oct-Nov)", 11: "Post-monsoon (Oct-Nov)"})
display(se.groupby(["season", "vehicle_class"]).service_failure.mean().unstack().style.format("{:.1%}"))

**Queue wait is not where the experience breaks.** Mean wait for completed swaps is ~243 s in every hour, city and station generation, with no link to load. Failure shows up as *no battery at all*, not as a longer queue. Abandoned attempts are the exception: riders give up after a median ~11 minutes.

In [ ]:
display(se[se.completed].groupby("charger_generation").queue_wait_sec.describe()[["mean", "50%", "75%"]])
display(se.groupby("event_type", observed=True).queue_wait_sec.median().rename("median wait (s)").to_frame().T)
print("Correlation of station-day mean wait with station-day failure rate:",
      round(se.groupby(["station_id", "date"], observed=True).agg(w=("queue_wait_sec", "mean"), f=("service_failure", "mean")).corr().iloc[0, 1], 3))

### Is failure concentrated?

In [ ]:
sf = se.groupby("station_id", observed=True).agg(attempts=("event_id", "size"), failures=("service_failure", "sum"), fail=("service_failure", "mean")).join(
    st.set_index("station_id")[["city", "charger_generation", "location_type", "expansion_wave", "connectivity_tier"]])
sf = sf.sort_values("fail", ascending=False)
lor = sf.sort_values("failures", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.3))
ax = axes[0]
ax.plot(np.arange(1, len(lor) + 1) / len(lor), lor.failures.cumsum() / lor.failures.sum(), color=PAL["bad"], lw=2.2, label="failures")
ax.plot(np.arange(1, len(lor) + 1) / len(lor), lor.attempts.cumsum() / lor.attempts.sum(), color=PAL["primary"], lw=2.2, label="attempts (same stations)")
ax.plot([0, 1], [0, 1], color=PAL["muted"], ls=":")
pct(ax); pct(ax, "x"); ax.legend(); ax.set_xlabel("share of stations (worst first)"); ax.set_title("Top 20% of stations: 43% of failures, 31% of attempts")
ax = axes[1]
sns.stripplot(data=sf.reset_index(), x="city", y="fail", hue="charger_generation", palette=GEN_PAL, ax=ax, size=6, jitter=.25,
              order=["Jaipur", "Delhi NCR", "Hyderabad", "Pune", "Bengaluru", "Mumbai"])
pct(ax, decimals=0); ax.set_title("Station failure rate: the high tail is Gen1 in three cities"); ax.set_xlabel("")
save(fig, "06_station_concentration")
top = sf.head(25)
print(f"25 worst stations: {top.charger_generation.value_counts().to_dict()} | {top.expansion_wave.value_counts().to_dict()} | {top.city.value_counts().to_dict()}")
k = int(len(lor) * .2)
print(f"Top 20% of stations by failures hold {lor.failures.iloc[:k].sum() / lor.failures.sum():.1%} of failures and {lor.attempts.iloc[:k].sum() / lor.attempts.sum():.1%} of attempts")

In [ ]:
ct = pd.crosstab(se.charger_generation == "Gen1", se.service_failure)
chi2, pval, *_ = stats.chi2_contingency(ct)
g1 = se[se.charger_generation == "Gen1"].service_failure.mean()
g23 = se[se.charger_generation != "Gen1"].service_failure.mean()
print(f"Gen1 failure {g1:.2%} vs Gen2/3 {g23:.2%}  (chi-square={chi2:,.0f}, p={pval:.1e})")

### Answer to Q2
* **Not evenly spread.** Failures concentrate at **Gen1 charger stations in Delhi NCR, Jaipur and Hyderabad**. All 25 worst stations are Gen1 launch-era sites in those three cities (13 Delhi NCR, 10 Jaipur, 2 Hyderabad); the worst 20% of stations carry 43% of failures on 31% of attempts. Gen1 fails at 6.8% vs 4.4% for Gen2/3 (p ≈ 0).
* **Season beats hour.** Evening peaks (19–22 h) fail slightly more (6.2% vs ~5.0%), but the dominant pattern is **summer**: failure more than doubles from April to June, and the extra failures are stockouts plus the abandonment that follows.
* **3W riders fail twice as often as 2W riders** in every month (8–15% vs 3–11%). 3W slots are scarce, and 3W riders cannot fall back on 2W stock.
* **Queue wait is flat** (~4 minutes) and uninformative. The customer experience fails through *empty cabinets*, not slow queues.